# Fase 3: Diseño algorítmico y eficiencia computacional

## Proyecto transversal

### Análisis de procedimientos sancionatorios de la Superintendencia del Medio Ambiente (SMA)

---

**Integrantes:**

- Hernán Saavedra González
- Lenin Morales Salas
- Rodrigo Gómez Indellicati


**Objetivo de la fase**

Diseñar, implementar y evaluar estrategias algorítmicas en Python para
el procesamiento y análisis del conjunto de datos de procedimientos
sancionatorios de la SMA, considerando criterios de eficiencia temporal,
uso de memoria, modularidad y reproducibilidad.

**Repositorio del proyecto:**

GitHub - proyecto_sma

## 1. Contexto del análisis

El presente notebook corresponde a la Fase 3 del proyecto transversal y continúa
el trabajo desarrollado durante la Fase 2, donde se realizó la exploración,
limpieza y preparación inicial del conjunto de datos de procedimientos
sancionatorios de la Superintendencia del Medio Ambiente (SMA).

En esta etapa el foco se desplaza desde la preparación de datos hacia el diseño
algorítmico, evaluando estrategias de implementación en Python que permitan
procesar información de manera eficiente, modular y reproducible.

Los algoritmos desarrollados utilizarán como entrada el dataset procesado
obtenido en fases anteriores, manteniendo la separación entre datos originales,
datos transformados y código fuente.

La evaluación considerará criterios de:

- eficiencia temporal;
- consumo de memoria;
- claridad del código;
- reutilización mediante funciones y módulos.

## 2. Fuente de datos y reproducibilidad

El análisis utiliza como fuente el conjunto de datos de procedimientos
sancionatorios publicado por la Superintendencia del Medio Ambiente (SMA).

**Archivo utilizado:**

`Sancionatorios.xlsx`

**Características de la versión utilizada:**

- Fuente: Superintendencia del Medio Ambiente (SMA).
- Fecha de descarga: 09 de septiembre de 2026.
- Hora de descarga: 10:00 hrs.
- Ubicación de la copia original:
  `data/raw/Sancionatorios.xlsx`

La copia almacenada corresponde a la versión original utilizada durante el
proyecto y no debe ser reemplazada por nuevas descargas, debido a que la fuente
puede actualizarse periódicamente.

Para asegurar la reproducibilidad, las etapas posteriores trabajan sobre
versiones procesadas del dataset, manteniendo separación entre:

- datos originales (`data/raw`);
- datos procesados (`data/processed`);
- código fuente (`src`);
- notebooks de análisis.

Las decisiones de diseño algorítmico desarrolladas en esta fase consideran la discusión realizada en el foro técnico de la Semana 1, particularmente respecto de la necesidad de comparar implementaciones que resuelvan el mismo problema y de utilizar evidencia experimental para justificar decisiones de eficiencia.
A partir de esta discusión se decidió comparar implementaciones iterativas y vectorizadas para la normalización de RegionNombre, incorporando mediciones de tiempo y memoria, además de verificar la equivalencia funcional mediante assert.
También se consideró la posibilidad de utilizar programación recursiva. Debido a que el conjunto de datos presenta una estructura principalmente tabular y no jerárquica, se decidió evaluarla experimentalmente como alternativa, pero no utilizarla como estrategia principal.

In [138]:
# Librerías principales

import pandas as pd
import numpy as np

from pathlib import Path
import sys

In [139]:
from src.pipeline import Pipeline
from src.normalizacion_poo import (
    NormalizadorIterativo,
    NormalizadorVectorizado
)

In [140]:
# Configuración de rutas del proyecto

from pathlib import Path

RUTA_PROCESADO = Path("../data/processed/sancionatorios_procesados.csv")

RUTA_PROCESADO

WindowsPath('../data/processed/sancionatorios_procesados.csv')

In [141]:
# Carga del dataset procesado de Fase 2

df = pd.read_csv(RUTA_PROCESADO)

df.head()

,ProcesoSancionId,Expediente,ProcesoSancionTipoNombre,ProcesoSancionEstado,FechaInicio,FechaTermino,ConfirmaPdC,MultaTotalUTA,LinkSNIFA,UnidadFiscalizableId,Nombre,RegionNombre,ComunaNombre,Latitud,Longitud,CategoriaEconomicaNombre,SubCategoriaEconomicaNombre,LinkSNIFA_UF,FechaActualizacion,DuracionDias
0,2,F-002-2013,Fiscalización,Terminado - Sanción,2013-04-01,2013-08-13,No,94.0,https://snifa.sma.gob.cl/Sancionatorio/Ficha/2...,162,MALL PLAZA EGAÑA - LA REINA,Región Metropolitana,La Reina,-33.453000,-70.570000,Equipamiento,Centro comercial,http://snifa.sma.gob.cl/UnidadFiscalizable/Fic...,2026-09-12,134.0
1,4,D-004-2013,Denuncia,Terminado - Sanción,2013-05-02,2013-10-11,No,25.0,https://snifa.sma.gob.cl/Sancionatorio/Ficha/4...,2488,VERTEDERO DICHAM,Región de los Lagos,Chonchi,-42.627141,-73.841298,Saneamiento Ambiental,Vertedero,http://snifa.sma.gob.cl/UnidadFiscalizable/Fic...,2026-09-12,162.0
2,9,D-003-2013,Denuncia,Terminado - Sanción,2013-03-13,2013-08-19,No,12.0,https://snifa.sma.gob.cl/Sancionatorio/Ficha/9...,10211,ALIMENTOS RILA,Región de Arica y Parinacota,Arica,-18.398400,-70.290600,Agroindustrias,Planta procesadora de productos agrícolas,http://snifa.sma.gob.cl/UnidadFiscalizable/Fic...,2026-09-12,159.0
3,12,D-005-2013,Denuncia,Terminado - Sanción,2013-06-03,2015-03-26,No,1.0,https://snifa.sma.gob.cl/Sancionatorio/Ficha/1...,307,PLANTA PROCESADORA DE PRODUCTOS DEL MAR,Región de Arica y Parinacota,Arica,-18.394700,-70.294500,Pesca y Acuicultura,Elaboración y procesado de recursos hidrobioló...,http://snifa.sma.gob.cl/UnidadFiscalizable/Fic...,2026-09-12,661.0
4,13,F-007-2013,Fiscalización,Terminado - Sanción,2013-06-03,2013-11-27,No,865.0,https://snifa.sma.gob.cl/Sancionatorio/Ficha/1...,2435,REFINERIA ENAP BIOBIO,Región del Biobío,Hualpén,-36.775327,-73.116842,Instalación fabril,Planta de Refinería,http://snifa.sma.gob.cl/UnidadFiscalizable/Fic...,2026-09-12,177.0


In [142]:
# Prueba de las implementaciones POO mediante Pipeline

serie = df["RegionNombre"]

pipeline_iterativo = Pipeline(NormalizadorIterativo())
resultado_iterativo_poo = pipeline_iterativo.ejecutar(serie)

pipeline_vectorizado = Pipeline(NormalizadorVectorizado())
resultado_vectorizado_poo = pipeline_vectorizado.ejecutar(serie)

print("POO iterativa:")
print(resultado_iterativo_poo.head())

print("\nPOO vectorizada:")
print(resultado_vectorizado_poo.head())

POO iterativa:
0            Región Metropolitana
1             Región de los Lagos
2    Región de Arica y Parinacota
3    Región de Arica y Parinacota
4               Región del Biobío
Name: RegionNombre, dtype: object

POO vectorizada:
0            Región Metropolitana
1             Región de los Lagos
2    Región de Arica y Parinacota
3    Región de Arica y Parinacota
4               Región del Biobío
Name: RegionNombre, dtype: object


In [143]:
# Verificación de equivalencia entre las dos implementaciones POO

pd.testing.assert_series_equal(
    resultado_iterativo_poo,
    resultado_vectorizado_poo
)

print("✓ Las dos implementaciones POO producen resultados equivalentes.")

✓ Las dos implementaciones POO producen resultados equivalentes.


### Arquitectura orientada a objetos

La normalización de `RegionNombre` se implementa mediante una arquitectura orientada a objetos que separa la definición de la estrategia de normalización de su implementación concreta.

La clase abstracta `NormalizadorRegion` concentra las responsabilidades comunes, como la validación de la entrada y la construcción del diccionario de regiones oficiales. A partir de ella se implementan dos estrategias concretas:

- `NormalizadorIterativo`: recorre los valores de la serie uno a uno.
- `NormalizadorVectorizado`: utiliza operaciones vectorizadas de Pandas.

Esta separación permite comparar dos estrategias que resuelven el mismo problema manteniendo una interfaz común. De esta forma, el resto del notebook puede utilizar cualquiera de las implementaciones sin modificar la lógica que consume sus resultados.

La utilización de `Pipeline` permite además separar la ejecución del proceso de normalización de la implementación específica del normalizador. Esta decisión favorece la reutilización y facilita incorporar nuevas estrategias sin modificar el código existente.

Esta arquitectura puede escalar hacia la Fase 4 porque permite agregar nuevas estrategias de procesamiento, validación o análisis mediante nuevas clases y módulos, manteniendo separadas las responsabilidades y reduciendo el acoplamiento entre los componentes.

## Validación técnica y verificación

En esta sección se verifica el funcionamiento del pipeline orientado a objetos mediante tres escenarios: un caso normal utilizando el conjunto de datos real, un caso límite y un caso de excepción.

Estas pruebas permiten comprobar que las clases y el pipeline mantienen un comportamiento consistente y controlado ante diferentes condiciones de entrada.

In [144]:
# Caso normal: procesamiento de la columna RegionNombre

resultado_normal = pipeline_iterativo.ejecutar(df["RegionNombre"])

print("Caso normal ejecutado correctamente.")
print("Registros procesados:", len(resultado_normal))
print("Nombre de la serie:", resultado_normal.name)

Caso normal ejecutado correctamente.
Registros procesados: 984
Nombre de la serie: RegionNombre


### Caso límite: serie vacía

Se evalúa el comportamiento del normalizador cuando recibe una serie vacía, pero que mantiene la estructura esperada de una columna `RegionNombre`.

In [145]:
# Caso límite: serie vacía

serie_vacia = df["RegionNombre"].iloc[0:0]

resultado_vacio = pipeline_iterativo.ejecutar(serie_vacia)

print("Caso límite ejecutado correctamente.")
print("Registros procesados:", len(resultado_vacio))

Caso límite ejecutado correctamente.
Registros procesados: 0


In [146]:
assert len(resultado_vacio) == 0

print("✓ Validación del caso límite correcta.")

✓ Validación del caso límite correcta.


### Caso de excepción: entrada inválida

Se evalúa el comportamiento del normalizador frente a una entrada que no corresponde al tipo de dato esperado.

La clase base valida que la entrada sea una `pandas.Series`. Si se recibe otro tipo de objeto, se genera una excepción controlada.

In [147]:
# Caso de excepción: entrada inválida

try:
    pipeline_iterativo.ejecutar(None)

except TypeError as e:
    print("✓ Excepción controlada correctamente.")
    print("Tipo de excepción:", type(e).__name__)
    print("Mensaje:", str(e))

✓ Excepción controlada correctamente.
Tipo de excepción: TypeError
Mensaje: Se esperaba una pandas.Series y se recibió NoneType


In [148]:
# Verificación de que la excepción esperada fue generada

try:
    pipeline_iterativo.ejecutar(None)
    excepcion_generada = False

except TypeError:
    excepcion_generada = True

assert excepcion_generada

print("✓ La entrada inválida genera la excepción esperada.")

✓ La entrada inválida genera la excepción esperada.


## Eficiencia experimental

Se compara el comportamiento de las implementaciones iterativa y vectorizada de la normalización de `RegionNombre`.

La comparación considera tiempo de ejecución y consumo de memoria. Ambas implementaciones reciben la misma entrada y previamente se verificó que producen resultados equivalentes.

In [149]:
import timeit
import tracemalloc

In [150]:
normalizador_iterativo = NormalizadorIterativo()
normalizador_vectorizado = NormalizadorVectorizado()

serie_region = df["RegionNombre"]

In [151]:
# Medición de tiempo de ejecución

repeticiones = 10

tiempo_iterativo = timeit.timeit(
    lambda: normalizador_iterativo.normalizar(serie_region),
    number=repeticiones
)

tiempo_vectorizado = timeit.timeit(
    lambda: normalizador_vectorizado.normalizar(serie_region),
    number=repeticiones
)

promedio_iterativo = tiempo_iterativo / repeticiones
promedio_vectorizado = tiempo_vectorizado / repeticiones

print(f"Tiempo promedio iterativo: {promedio_iterativo:.8f} segundos")
print(f"Tiempo promedio vectorizado: {promedio_vectorizado:.8f} segundos")

Tiempo promedio iterativo: 0.00146449 segundos
Tiempo promedio vectorizado: 0.00162304 segundos


In [152]:
# Medición de consumo de memoria

def medir_memoria(funcion):
    tracemalloc.start()

    funcion()

    memoria_actual, memoria_maxima = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return memoria_maxima


memoria_iterativa = medir_memoria(
    lambda: normalizador_iterativo.normalizar(serie_region)
)

memoria_vectorizada = medir_memoria(
    lambda: normalizador_vectorizado.normalizar(serie_region)
)

print(f"Memoria máxima iterativa: {memoria_iterativa / 1024:.2f} KiB")
print(f"Memoria máxima vectorizada: {memoria_vectorizada / 1024:.2f} KiB")

Memoria máxima iterativa: 24.43 KiB
Memoria máxima vectorizada: 121.79 KiB


### Comportamiento según el tamaño de entrada

Para analizar por qué el comportamiento relativo de ambas implementaciones puede cambiar, se repite el experimento utilizando conjuntos de entrada de distinto tamaño.

Se consideran 984, 3.936 y 15.744 registros, manteniendo la misma estructura y distribución de valores de `RegionNombre`.

In [153]:
# Comparación con distintos tamaños de entrada

tamanos = [984, 3936, 15744]

resultados_tamanos = []

for tamano in tamanos:
    repeticiones = 10

    serie_prueba = pd.Series(
        pd.concat(
            [serie_region] * (tamano // len(serie_region) + 1),
            ignore_index=True
        ).iloc[:tamano]
    )

    tiempo_iter = timeit.timeit(
        lambda: normalizador_iterativo.normalizar(serie_prueba),
        number=repeticiones
    ) / repeticiones

    tiempo_vect = timeit.timeit(
        lambda: normalizador_vectorizado.normalizar(serie_prueba),
        number=repeticiones
    ) / repeticiones

    resultados_tamanos.append({
        "Registros": tamano,
        "Iterativa (s)": tiempo_iter,
        "Vectorizada (s)": tiempo_vect
    })

resultados_tiempos = pd.DataFrame(resultados_tamanos)

resultados_tiempos

,Registros,Iterativa (s),Vectorizada (s)
0,984,0.001402,0.001465
1,3936,0.005082,0.002898
2,15744,0.019547,0.009103


In [154]:
# Cálculo de la razón de tiempos

resultados_tiempos["Razón Iterativa/Vectorizada"] = (
    resultados_tiempos["Iterativa (s)"] /
    resultados_tiempos["Vectorizada (s)"]
)

resultados_tiempos

,Registros,Iterativa (s),Vectorizada (s),Razón Iterativa/Vectorizada
0,984,0.001402,0.001465,0.956808
1,3936,0.005082,0.002898,1.753865
2,15744,0.019547,0.009103,2.147271


### Interpretación del comportamiento según el tamaño de entrada

Los resultados experimentales muestran que el comportamiento relativo de las implementaciones depende del tamaño del conjunto de datos.

Para 984 registros, la implementación iterativa presentó un menor tiempo de ejecución (0,001624 s frente a 0,001920 s). En este caso, el costo asociado a las operaciones vectorizadas de Pandas representa una proporción relevante del tiempo total.

Al aumentar el conjunto a 3.936 registros, la implementación vectorizada presentó un menor tiempo de ejecución (0,004751 s frente a 0,006865 s). Para 15.744 registros, esta diferencia se hizo más evidente, con 0,009187 s para la versión vectorizada frente a 0,022767 s para la iterativa.

La razón entre ambos tiempos pasó de 0,8458 para 984 registros a 1,4476 para 3.936 registros y 2,4782 para 15.744 registros. Esto evidencia que la ventaja relativa de la vectorización aumenta a medida que crece el volumen de datos procesados.

Este comportamiento puede explicarse por la capacidad de Pandas para ejecutar operaciones sobre conjuntos de datos mediante operaciones internas optimizadas, evitando el recorrido explícito elemento por elemento de la implementación iterativa.

Los resultados corresponden a las condiciones específicas del experimento y no permiten establecer que una estrategia sea siempre superior a la otra. La elección depende del tamaño de los datos, las operaciones realizadas y los recursos utilizados.

In [155]:
# Dimensiones del dataset de entrada

df.shape

(984, 20)

In [156]:
# Información general del dataset

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 984 entries, 0 to 983
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   ProcesoSancionId             984 non-null    int64  
 1   Expediente                   984 non-null    str    
 2   ProcesoSancionTipoNombre     984 non-null    str    
 3   ProcesoSancionEstado         984 non-null    str    
 4   FechaInicio                  984 non-null    str    
 5   FechaTermino                 982 non-null    str    
 6   ConfirmaPdC                  984 non-null    str    
 7   MultaTotalUTA                984 non-null    float64
 8   LinkSNIFA                    984 non-null    str    
 9   UnidadFiscalizableId         984 non-null    int64  
 10  Nombre                       984 non-null    str    
 11  RegionNombre                 984 non-null    str    
 12  ComunaNombre                 984 non-null    str    
 13  Latitud                      97

## 3. Diseño algorítmico propuesto

A partir del conjunto de datos procesado en la Fase 2, se diseñan algoritmos orientados a generar información analítica sobre los procedimientos sancionatorios de la Superintendencia del Medio Ambiente (SMA).

El diseño considera una separación entre la lógica de procesamiento y la visualización de resultados, utilizando funciones reutilizables almacenadas en módulos Python dentro de la carpeta `src/`.

Considerando las características del dataset de entrada, las decisiones algorítmicas se orientan a construir soluciones modulares, reproducibles y eficientes.

El problema no presenta una alta magnitud de registros, por lo que la mejora computacional se enfocará principalmente en:

- organización del procesamiento;
- reutilización de funciones;
- reducción de operaciones repetidas;
- claridad y mantenimiento del código.

Se evaluarán dos estrategias de implementación:

## Programación estructurada

La estrategia principal utilizada en el proyecto corresponde a programación estructurada mediante funciones independientes, donde cada función representa una operación específica del procesamiento analítico.

Este enfoque permite separar responsabilidades y facilitar la reutilización del código. Las principales operaciones consideradas son:

- construcción del universo analítico;
- cálculo de indicadores asociados a procedimientos sancionatorios;
- cálculo de duración de procesos;
- normalización y transformación de variables.

La implementación mediante funciones permite que el notebook actúe como controlador del flujo de análisis, mientras que la lógica de procesamiento permanece encapsulada dentro de módulos Python ubicados en la carpeta `src/`.

Desde el punto de vista de eficiencia temporal, se priorizan operaciones vectorizadas mediante Pandas, evitando iteraciones manuales innecesarias sobre los registros del DataFrame.

## Programación recursiva

Las funciones recursivas son una alternativa útil cuando el problema presenta una estructura jerárquica o requiere dividir un problema en subproblemas equivalentes.

Sin embargo, para el análisis de procedimientos sancionatorios de la SMA, los datos presentan una estructura tabular donde predominan operaciones de agrupación, transformación y cálculo de indicadores.

Por esta razón, se evaluará críticamente su aplicación, considerando que una solución recursiva podría aumentar el consumo de memoria debido a la utilización de la pila de llamadas, sin entregar ventajas significativas frente a las operaciones vectorizadas disponibles en Pandas.

La selección del enfoque algorítmico se realizará considerando:

- tiempo de ejecución;
- consumo de memoria;
- claridad del código;
- facilidad de mantenimiento.

# 4. Implementación modular de algoritmos

Para mejorar la mantenibilidad y reproducibilidad del proyecto, la lógica de procesamiento se separa del notebook principal mediante módulos Python ubicados en la carpeta `src/`.

Esta organización permite que el notebook actúe como controlador del flujo de análisis, mientras que las funciones de procesamiento pueden ser reutilizadas, probadas y versionadas de manera independiente.

La estructura implementada es:

src/
├── procesamiento.py
└── mediciones.py

Cada módulo tiene una responsabilidad específica:

- `procesamiento.py`: contiene funciones relacionadas con preparación y transformación de datos.
- `mediciones.py`: contiene funciones para evaluar tiempo de ejecución y eficiencia computacional.

Esta separación facilita la trazabilidad del código y permite evaluar el desempeño de los algoritmos desarrollados.

In [157]:
from pathlib import Path
import sys

# Obtener raíz del proyecto
RAIZ_PROYECTO = Path.cwd().parent

# Agregar raíz al path de Python
sys.path.append(str(RAIZ_PROYECTO))

RAIZ_PROYECTO

WindowsPath('c:/Users/Papá/Documents/proyecto_sma')

In [158]:
from src.procesamiento import construir_universo_analitico
from src.procesamiento import calcular_duracion_proceso

In [159]:
universo = construir_universo_analitico(df)

universo.head()

,UnidadFiscalizableId,cantidad_procesos,multa_total
0,155,1,9.5
1,156,1,139.2
2,162,2,235.0
3,220,1,304.0
4,227,1,60.0


### Resultado del algoritmo: construcción del universo analítico

La función `construir_universo_analitico()` permite transformar el dataset de procedimientos sancionatorios en una estructura resumida por unidad fiscalizable.

El algoritmo utiliza operaciones de agrupación mediante Pandas (`groupby`), permitiendo obtener indicadores agregados como:

- cantidad de procedimientos asociados a una unidad fiscalizable;
- monto total acumulado de multas.

La estrategia utilizada corresponde a un enfoque estructurado y vectorizado, evitando recorrer manualmente cada registro mediante ciclos, lo que mejora la legibilidad y aprovecha las optimizaciones internas de la biblioteca Pandas.

Desde el punto de vista de complejidad, la operación presenta un comportamiento aproximado O(n), debido a que requiere procesar los registros del DataFrame de entrada para generar los agrupamientos correspondientes.

In [160]:
datos_duracion = calcular_duracion_proceso(df)

datos_duracion[
    ["ProcesoSancionId", "FechaInicio", "FechaTermino", "DuracionDias"]
].head()

,ProcesoSancionId,FechaInicio,FechaTermino,DuracionDias
0,2,2013-04-01,2013-08-13,134.0
1,4,2013-05-02,2013-10-11,162.0
2,9,2013-03-13,2013-08-19,159.0
3,12,2013-06-03,2015-03-26,661.0
4,13,2013-06-03,2013-11-27,177.0


### Resultado del algoritmo: cálculo de duración de procedimientos

La función `calcular_duracion_proceso()` permite obtener la duración temporal de cada procedimiento sancionatorio mediante la diferencia entre la fecha de inicio y la fecha de término.

Este procesamiento genera una nueva variable denominada `DuracionDias`, la cual permite analizar posteriormente el comportamiento temporal de los procesos sancionatorios.

La implementación utiliza operaciones vectorizadas de Pandas para transformar las fechas y calcular diferencias entre columnas, evitando iteraciones registro por registro.

Desde la perspectiva de eficiencia algorítmica, la operación presenta una complejidad aproximada O(n), debido a que cada registro del conjunto de datos es procesado una única vez.

El uso de funciones independientes favorece la modularidad del proyecto, permitiendo reutilizar este cálculo en diferentes etapas del análisis sin duplicar código.

## 5. Medición de eficiencia computacional

Para evaluar empíricamente el comportamiento de los algoritmos desarrollados, se incorporará una medición del tiempo de ejecución.

La medición se realizará utilizando `time.perf_counter()`, que permite registrar el tiempo transcurrido durante la ejecución de una función.

El objetivo es complementar el análisis teórico de complejidad con una medición observable sobre el conjunto de datos utilizado en el proyecto.

In [161]:
from src.mediciones import medir_tiempo

In [162]:
universo, tiempo_universo = medir_tiempo(
    construir_universo_analitico,
    df
)

print(f"Tiempo de ejecución: {tiempo_universo:.6f} segundos")

Tiempo de ejecución: 0.004145 segundos


In [163]:
datos_duracion, tiempo_duracion = medir_tiempo(
    calcular_duracion_proceso,
    df
)

print(f"Tiempo de ejecución: {tiempo_duracion:.6f} segundos")

Tiempo de ejecución: 0.003431 segundos


## 5.1. Comparación de tiempos de ejecución

Las mediciones repetidas realizadas sobre el conjunto de datos utilizado muestran que la construcción del universo analítico presentó un tiempo promedio de 0,003444 segundos, mientras que el cálculo de duración de los procedimientos presentó un tiempo promedio de 0,002916 segundos.
La diferencia observada se relaciona con las operaciones que realiza cada algoritmo. La construcción del universo analítico utiliza operaciones de agrupación mediante groupby() y agregaciones, mientras que el cálculo de duración realiza principalmente transformaciones de fechas y operaciones vectorizadas.
Estos valores corresponden a las condiciones específicas del experimento sobre un conjunto de 984 registros y deben interpretarse como una referencia experimental, no como una medida general del rendimiento de los algoritmos.
Ambos procedimientos presentan una complejidad temporal aproximada O(n), ya que el procesamiento depende del número de registros de entrada.

In [164]:
import time

def medir_repetidamente(funcion, datos, repeticiones=10):
    tiempos = []

    for _ in range(repeticiones):
        inicio = time.perf_counter()
        funcion(datos)
        fin = time.perf_counter()

        tiempos.append(fin - inicio)

    return {
        "minimo": min(tiempos),
        "maximo": max(tiempos),
        "promedio": sum(tiempos) / len(tiempos)
    }

In [165]:
resultado_universo = medir_repetidamente(
    construir_universo_analitico,
    df
)

resultado_universo

{'minimo': 0.002580599975772202,
 'maximo': 0.0042212000116705894,
 'promedio': 0.002897280000615865}

In [166]:
resultado_duracion = medir_repetidamente(
    calcular_duracion_proceso,
    df
)

resultado_duracion

{'minimo': 0.0023905999260023236,
 'maximo': 0.0051386000122874975,
 'promedio': 0.0029530299943871796}

In [167]:
comparacion_tiempos = pd.DataFrame({
    "Algoritmo": [
        "Construcción del universo analítico",
        "Cálculo de duración de procesos"
    ],
    "Tiempo mínimo (s)": [
        resultado_universo["minimo"],
        resultado_duracion["minimo"]
    ],
    "Tiempo máximo (s)": [
        resultado_universo["maximo"],
        resultado_duracion["maximo"]
    ],
    "Tiempo promedio (s)": [
        resultado_universo["promedio"],
        resultado_duracion["promedio"]
    ]
})

comparacion_tiempos

,Algoritmo,Tiempo mínimo (s),Tiempo máximo (s),Tiempo promedio (s)
0,Construcción del universo analítico,0.002581,0.004221,0.002897
1,Cálculo de duración de procesos,0.002391,0.005139,0.002953


## 5.2. Análisis de los resultados

Las mediciones repetidas permiten complementar el análisis teórico de complejidad con evidencia experimental.

Las mediciones repetidas permiten complementar el análisis teórico de complejidad con evidencia experimental.  
Para la construcción del universo analítico se obtuvo un tiempo promedio de 0,003444 segundos, mientras que el cálculo de duración de los procesos presentó un promedio de 0,002916 segundos.
La diferencia observada es pequeña y debe interpretarse considerando que ambos algoritmos realizan operaciones diferentes. La construcción del universo analítico utiliza operaciones de agrupación mediante groupby() y operaciones de agregación, mientras que el cálculo de duración realiza transformaciones de fechas y operaciones vectorizadas.
En ambos casos, el procesamiento depende del número de registros de entrada, por lo que se considera una complejidad temporal aproximada O(n). Los tiempos obtenidos corresponden al dataset actual de 984 registros y constituyen una línea base para futuras mediciones.
La repetición de las pruebas permite reducir la dependencia de una única ejecución y proporciona una referencia más estable para comparar modificaciones posteriores del código.

La diferencia observada es pequeña y debe interpretarse considerando que ambos algoritmos realizan operaciones diferentes. La construcción del universo analítico utiliza agrupación mediante `groupby()` y operaciones de agregación, mientras que el cálculo de duración realiza transformaciones de fechas y operaciones vectorizadas.

En ambos casos, el procesamiento depende del número de registros de entrada, por lo que se considera una complejidad temporal aproximada O(n). Los tiempos obtenidos corresponden al dataset actual de 984 registros y constituyen una línea base para futuras mediciones.

La repetición de las pruebas permite reducir la dependencia de una única ejecución y proporciona una referencia más estable para comparar modificaciones posteriores del código.

In [168]:
import tracemalloc


def medir_memoria(funcion, datos):
    tracemalloc.start()

    resultado = funcion(datos)

    memoria_actual, memoria_pico = tracemalloc.get_traced_memory()

    tracemalloc.stop()

    return resultado, memoria_pico

In [169]:
universo_memoria, memoria_universo = medir_memoria(
    construir_universo_analitico,
    df
)

print(f"Memoria máxima utilizada: {memoria_universo / 1024:.2f} KB")

Memoria máxima utilizada: 75.57 KB


In [170]:
duracion_memoria, memoria_duracion = medir_memoria(
    calcular_duracion_proceso,
    df
)

print(f"Memoria máxima utilizada: {memoria_duracion / 1024:.2f} KB")

Memoria máxima utilizada: 235.54 KB


In [171]:
comparacion_eficiencia = pd.DataFrame({
    "Algoritmo": [
        "Construcción del universo analítico",
        "Cálculo de duración de procesos"
    ],
    "Tiempo promedio (s)": [
        resultado_universo["promedio"],
        resultado_duracion["promedio"]
    ],
    "Memoria máxima (KB)": [
        memoria_universo / 1024,
        memoria_duracion / 1024
    ]
})

comparacion_eficiencia

,Algoritmo,Tiempo promedio (s),Memoria máxima (KB)
0,Construcción del universo analítico,0.002897,75.571289
1,Cálculo de duración de procesos,0.002953,235.537109


### 5.3. Análisis conjunto de tiempo y memoria

La comparación permite observar que ambos algoritmos presentan tiempos de ejecución bajos para el conjunto de datos utilizado. La construcción del universo analítico presentó un tiempo promedio de 0,003444 segundos, mientras que el cálculo de duración de los procedimientos presentó un promedio de 0,002916 segundos.
En términos de memoria, se observa un comportamiento diferente. La construcción del universo analítico registró un consumo máximo aproximado de 75,96 KB, mientras que el cálculo de duración alcanzó aproximadamente 235,65 KB.
Esta diferencia se relaciona con la implementación utilizada. La función calcular_duracion_proceso() realiza transformaciones sobre las columnas de fecha y genera estructuras temporales adicionales. Por lo tanto, un menor tiempo de ejecución no implica necesariamente un menor consumo de memoria.
Para ambos algoritmos se considera un comportamiento temporal aproximado O(n), dado que el procesamiento depende del número de registros de entrada. El consumo de memoria depende además de las estructuras temporales generadas durante las operaciones.
Los resultados obtenidos constituyen una línea base experimental para comparar futuras mejoras de implementación.

## 5.4.Justificación del uso de programación estructurada y no recursiva

En el conjunto de datos analizado no se identifica una estructura jerárquica o una dependencia entre registros que requiera resolver el problema mediante llamadas recursivas. El procesamiento corresponde principalmente a operaciones sobre un DataFrame, transformaciones de columnas, normalización de valores y agregaciones.
Por esta razón, la recursividad no se incorpora como estrategia de implementación para los procedimientos sancionatorios analizados. Su utilización únicamente para cumplir con el requisito de evaluar una estrategia recursiva introduciría una solución artificial y no representativa del problema.
En consecuencia, para este proyecto se prioriza la programación estructurada mediante funciones independientes y las operaciones vectorizadas de Pandas. Esta decisión permite mantener una correspondencia entre la estrategia algorítmica y la naturaleza tabular de los datos, además de facilitar la medición de tiempo, consumo de memoria y validación de resultados.

## 5.4.1. Evaluación de programación recursiva

Como parte de las estrategias algorítmicas evaluadas durante la Fase 3, se consideró la utilización de programación recursiva. Sin embargo, el conjunto de datos analizado presenta una estructura principalmente tabular, donde las operaciones corresponden a transformaciones, agrupaciones, normalización y cálculo de indicadores sobre registros.
Debido a estas características, no se identifica una estructura jerárquica que justifique utilizar la recursividad como estrategia principal de procesamiento. Por esta razón, se optó por implementar el procesamiento mediante funciones estructuradas y operaciones vectorizadas de Pandas.
La recursividad fue considerada como alternativa algorítmica, pero su incorporación al procesamiento principal no entregaría una ventaja significativa y podría aumentar el consumo de memoria debido a la utilización de la pila de llamadas.
En el experimento realizado, la implementación recursiva alcanzó 86,05 KB, frente a 75,45 KB de la implementación iterativa.

In [172]:
def contar_por_unidad_recursivo(valores, indice=0, conteo=None):
    if conteo is None:
        conteo = {}

    if indice >= len(valores):
        return conteo

    unidad = valores.iloc[indice]

    if unidad in conteo:
        conteo[unidad] += 1
    else:
        conteo[unidad] = 1

    return contar_por_unidad_recursivo(valores, indice + 1, conteo)

In [173]:
import sys

sys.setrecursionlimit(5000)

In [174]:
resultado_recursivo = contar_por_unidad_recursivo(
    df["UnidadFiscalizableId"].reset_index(drop=True)
)

print("Primeros resultados - versión recursiva:")
print(list(resultado_recursivo.items())[:5])

Primeros resultados - versión recursiva:
[(np.int64(162), 2), (np.int64(2488), 2), (np.int64(10211), 1), (np.int64(307), 1), (np.int64(2435), 1)]


In [175]:
def contar_por_unidad_iterativo(df):
    conteo = {}

    for unidad in df["UnidadFiscalizableId"]:
        if unidad in conteo:
            conteo[unidad] += 1
        else:
            conteo[unidad] = 1

    return conteo

In [176]:
def contar_por_unidad_pandas(df):
    return df["UnidadFiscalizableId"].value_counts().to_dict()

In [177]:
resultado_iterativo = contar_por_unidad_iterativo(df)
resultado_pandas = contar_por_unidad_pandas(df)

print("Primeros resultados - versión iterativa:")
print(list(resultado_iterativo.items())[:5])

print("\nPrimeros resultados - versión Pandas:")
print(list(resultado_pandas.items())[:5])

Primeros resultados - versión iterativa:
[(162, 2), (2488, 2), (10211, 1), (307, 1), (2435, 1)]

Primeros resultados - versión Pandas:
[(1381, 3), (13738, 3), (13957, 3), (162, 2), (2488, 2)]


In [178]:
assert resultado_recursivo == resultado_iterativo

print("✓ La implementación recursiva produce el mismo resultado.")

✓ La implementación recursiva produce el mismo resultado.


La implementación recursiva permite obtener el mismo resultado que la versión iterativa. Sin embargo, al procesar el conjunto completo de 984 registros fue necesario aumentar el límite de profundidad de recursión de Python. Esto evidencia una limitación práctica de este enfoque para procesar grandes cantidades de registros, ya que cada llamada recursiva utiliza espacio en la pila de ejecución.

In [179]:
assert resultado_iterativo == resultado_pandas

print("✓ Ambas implementaciones producen el mismo resultado.")

✓ Ambas implementaciones producen el mismo resultado.


In [180]:
print("Versión recursiva:")
print(resultado_tiempo_recursivo)

print("\nVersión iterativa:")
print(resultado_tiempo_iterativo)

print("\nVersión Pandas:")
print(resultado_tiempo_pandas)

Versión recursiva:
{'minimo': 0.005073000094853342, 'maximo': 0.0061144999926909804, 'promedio': 0.005331050010863691}

Versión iterativa:
{'minimo': 0.0001669999910518527, 'maximo': 0.0002443999983370304, 'promedio': 0.00017953000497072935}

Versión Pandas:
{'minimo': 0.000358999939635396, 'maximo': 0.0005898999515920877, 'promedio': 0.0003963299910537899}


In [181]:
resultado_tiempo_recursivo = medir_repetidamente(
    contar_por_unidad_recursivo,
    df["UnidadFiscalizableId"].reset_index(drop=True)
)

resultado_tiempo_iterativo = medir_repetidamente(
    contar_por_unidad_iterativo,
    df
)

resultado_tiempo_pandas = medir_repetidamente(
    contar_por_unidad_pandas,
    df
)

print("Versión recursiva:")
print(resultado_tiempo_recursivo)

print("\nVersión iterativa:")
print(resultado_tiempo_iterativo)

print("\nVersión Pandas:")
print(resultado_tiempo_pandas)

Versión recursiva:
{'minimo': 0.005080300034023821, 'maximo': 0.005470899981446564, 'promedio': 0.005246889998670667}

Versión iterativa:
{'minimo': 0.00016609998419880867, 'maximo': 0.0002509999321773648, 'promedio': 0.00018011999782174826}

Versión Pandas:
{'minimo': 0.00036339997313916683, 'maximo': 0.0005916999652981758, 'promedio': 0.00040112999267876147}


In [182]:
_, memoria_recursiva_unidad = medir_memoria(
    lambda datos: contar_por_unidad_recursivo(datos),
    df["UnidadFiscalizableId"].reset_index(drop=True)
)

_, memoria_iterativa_unidad = medir_memoria(
    lambda datos: contar_por_unidad_iterativo(datos),
    df
)

_, memoria_pandas_unidad = medir_memoria(
    lambda datos: contar_por_unidad_pandas(datos),
    df
)

print(f"Memoria versión recursiva: {memoria_recursiva_unidad / 1024:.2f} KB")
print(f"Memoria versión iterativa: {memoria_iterativa_unidad / 1024:.2f} KB")
print(f"Memoria versión Pandas: {memoria_pandas_unidad / 1024:.2f} KB")

Memoria versión recursiva: 84.89 KB
Memoria versión iterativa: 75.45 KB
Memoria versión Pandas: 92.00 KB


## Análisis de la evaluación recursiva
La implementación recursiva permitió obtener resultados equivalentes a las versiones iterativa y vectorizada. Sin embargo, su ejecución requirió aumentar el límite de profundidad de recursión de Python para procesar los 984 registros del conjunto de datos.
Además, las mediciones experimentales permiten comparar el comportamiento de las tres estrategias en términos de tiempo de ejecución y consumo de memoria. La implementación recursiva incorpora una llamada adicional por cada registro procesado, lo que aumenta su dependencia de la pila de ejecución. En este contexto, la programación iterativa y las operaciones vectorizadas de Pandas resultan más adecuadas para el procesamiento del conjunto de datos analizado.
Por estas razones, la recursividad se mantiene como una alternativa evaluada experimentalmente, pero no se utiliza como estrategia principal del proyecto.

## 5.5 Comparación experimental de normalización de `RegionNombre`

Para complementar el análisis de eficiencia computacional, se realizó una comparación experimental entre una implementación iterativa y una implementación vectorizada mediante operaciones de Pandas para la normalización de la variable `RegionNombre`.

Ambas implementaciones fueron aplicadas sobre la misma serie de entrada RegionNombre y se verificó mediante assert que producen resultados equivalentes. Ambas implementaciones fueron aplicadas a la misma entrada y se verificó mediante assert que producen resultados equivalentes. Esta comprobación permite asegurar que la optimización no modifica el resultado funcional del procesamiento.

La evaluación se realizó utilizando distintos tamaños de entrada, con el objetivo de observar el comportamiento de ambas estrategias a medida que aumenta el número de registros.

### Resultados de tiempo de ejecución

| Registros | Iterativa (ms) | Vectorizada (ms) | Razón de tiempo |
|---:|---:|---:|---:|
| 984 | 0,50 | 0,62 | 0,81 |
| 3.936 | 1,96 | 1,61 | 1,22 |
| 15.744 | 7,77 | 5,14 | 1,51 |

Para 984 registros, la implementación iterativa presentó un menor tiempo de ejecución. Sin embargo, al aumentar el tamaño de entrada a 3.936 y 15.744 registros, la implementación vectorizada presentó menores tiempos de ejecución.

La razón de tiempo permite observar este comportamiento. Un valor superior a 1 indica que, para esa prueba, la implementación vectorizada presentó un menor tiempo de ejecución respecto de la implementación iterativa.

### Uso de memoria

| Registros | Iterativa (KiB) | Vectorizada (KiB) |
|---:|---:|---:|
| 984 | 24,4 | 105,4 |
| 3.936 | 94,1 | 404,3 |
| 15.744 | 379,8 | 1.600,3 |

En términos de memoria, la implementación iterativa presentó un menor consumo máximo en las tres pruebas realizadas. El consumo aumenta a medida que crece el número de registros procesados, observándose una diferencia mayor entre ambas estrategias para los tamaños de entrada más grandes.

Estos resultados corresponden a las condiciones específicas del experimento y no deben interpretarse como una conclusión general aplicable a cualquier algoritmo o conjunto de datos.

### Equivalencia funcional

La equivalencia funcional entre ambas implementaciones fue comprobada mediante `assert`, verificando que producen los mismos resultados para el conjunto de datos utilizado.

Además, el resultado de la normalización fue comparado con el archivo CSV generado durante la Fase 2, confirmando la correspondencia de los resultados.

### Análisis experimental

La comparación experimental permite complementar el análisis teórico de complejidad realizado durante la Fase 3. Los resultados muestran que el comportamiento relativo de ambas implementaciones depende del tamaño del conjunto de datos.
Para 984 registros, la implementación iterativa presentó un menor tiempo de ejecución que la versión vectorizada. Sin embargo, al aumentar la cantidad de registros a 3.936 y 15.744, la implementación vectorizada presentó menores tiempos de ejecución.
Este comportamiento puede explicarse por el costo fijo asociado a las operaciones de Pandas. Para conjuntos pequeños, dicho costo puede representar una proporción importante del tiempo total y una implementación iterativa simple puede resultar competitiva. A medida que aumenta el número de registros, las operaciones vectorizadas permiten procesar los datos mediante operaciones internas optimizadas, reduciendo el costo relativo de recorrer los registros individualmente.
En términos de memoria, la implementación iterativa presentó un menor consumo máximo en las tres pruebas realizadas. Por lo tanto, la evaluación de eficiencia debe considerar conjuntamente el tiempo de ejecución y el uso de memoria.
Los resultados corresponden a las condiciones específicas del experimento y constituyen evidencia experimental para complementar el análisis de complejidad y establecer una línea base para futuras comparaciones.

In [183]:
# Caso límite: serie vacía

serie_vacia = pd.Series([], dtype="object", name="RegionNombre")

resultado_limite = pipeline_iterativo.ejecutar(serie_vacia)

print("Caso límite ejecutado correctamente.")
print("Cantidad de registros:", len(resultado_limite))

assert len(resultado_limite) == 0

Caso límite ejecutado correctamente.
Cantidad de registros: 0


In [184]:
# Caso de excepción: región no reconocida

serie_invalida = pd.Series(
    ["Región Inexistente"],
    name="RegionNombre"
)

try:
    pipeline_iterativo.ejecutar(serie_invalida)
except ValueError as e:
    print("Caso de excepción controlado correctamente.")
    print("Mensaje:", e)
else:
    raise AssertionError("Se esperaba un ValueError.")

Caso de excepción controlado correctamente.
Mensaje: Región no reconocida: 'Región Inexistente'


# 6. Conclusiones

La Fase 3 permitió diseñar, implementar y evaluar distintas estrategias algorítmicas aplicadas al conjunto de datos de procedimientos sancionatorios de la Superintendencia del Medio Ambiente.

A partir de la implementación de funciones modulares en Python, se logró separar la lógica de procesamiento, los algoritmos y las funciones destinadas a medir su eficiencia. Esta organización favorece la reutilización, trazabilidad y mantenimiento del código.

Las mediciones experimentales permitieron complementar el análisis teórico de complejidad. Se observaron diferencias en los tiempos de ejecución y en el consumo de memoria, dependiendo de las operaciones realizadas, el tamaño del conjunto de datos y la estrategia de implementación utilizada.

La comparación entre implementaciones iterativas y vectorizadas permitió comprobar que ambas estrategias pueden producir resultados equivalentes, pero presentar diferentes comportamientos computacionales. Las pruebas realizadas con distintos tamaños de entrada mostraron que el comportamiento relativo de las implementaciones puede variar a medida que aumenta la cantidad de registros procesados.

La evaluación de alternativas algorítmicas permitió determinar que la recursividad no resulta necesaria para el problema analizado, debido a que los procedimientos trabajados corresponden principalmente a transformaciones y operaciones sobre estructuras tabulares. En consecuencia, se priorizaron estrategias iterativas y vectorizadas, cuya aplicación resulta más coherente con las características del conjunto de datos.

La verificación mediante `assert` y la comparación con los resultados obtenidos durante la Fase 2 permitieron comprobar la equivalencia funcional de las implementaciones y mantener la trazabilidad de los resultados.

En conjunto, las experiencias desarrolladas permiten establecer una línea base experimental para futuras mejoras del proyecto y muestran la importancia de considerar simultáneamente corrección, complejidad, tiempo de ejecución, consumo de memoria, modularidad y reproducibilidad al evaluar distintas estrategias algorítmicas.